In [ ]:
import jax
import master.device as device
from master.obj import *
from master.estimator_bounds import *
import numpy as np
from master import *

jax.config.update("jax_enable_x64", True)
jax.devices()
device_id = 0
device.setup_device(device_id=device_id)

In [ ]:
base_path = "cdi-128/"
loader = Loader(base_path)

fig_path = "../docs/thesis/figures"
SAVE_FIGS = True

In [ ]:
pixel_intensity = 1e3
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
est_id = "ORACLE-averaged-second-order-ml"
est_label = "ORACLE+FS"

x_true, d, measurement_params, measurement_hash_meas = loader.get_measurement(meas_id)
x_est, simulation_params, lls, config_hash, measurement_hash_est = loader.get_estimate(
    meas_id, est_id
)

# Check that measurement corresponds to reconstruction
assert measurement_hash_est == measurement_hash_meas
sim_params = simulation_params["simulation"]

N_crop = sim_params["N_crop"]
N_pad = sim_params["N_pad"]
N_photons = sim_params["N_photons"]
x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = create_physical_axes(
    x_true, pixel_size=1
)
r_extent_cropped = list(
    create_physical_axes(center_crop(x_true, N_crop), pixel_size=1)
)[-2]
print(k_extent)

x_true.shape, d.shape, x_est.shape, simulation_params["simulation"]

In [ ]:
loader_2 = Loader("cdi-128-ORACLE/")
x_true_2, d_2, measurement_params_2, measurement_hash_meas_2 = loader.get_measurement(
    meas_id
)
# Sanity check: the data of the oracle set and the HIO ones should be the same
assert jnp.allclose(d, d_2, rtol=1e-12, atol=1e-14)

# Ground truth object

In [ ]:
x_cameraman = cameraman() * 0.9 + 0.1
x_mandrill = as_phase_field(mandrill() ** 2 - 0.5)
x_obj, obj_support = crop_and_pad(x_cameraman * x_mandrill, N_crop=N_crop, N_pad=N_pad)
x_obj = normalize_image_intensity(x_obj)
x_new_true = x_obj * jnp.sqrt(N_photons)
assert jnp.allclose(x_new_true, x_true, rtol=1e-12, atol=1e-14)
if SAVE_FIGS:
    save_path = f"{fig_path}/cdi_object.png"
else:
    save_path = None

x_obj_transfer = x_obj / jnp.max(jnp.abs(x_obj))
T_factor = 1 / jnp.max(jnp.abs(x_true))
plot_intensity_phase(x_obj_transfer, save_path=save_path, extent=r_extent)

In [ ]:
print(rf"{65536000.0:.3g}")

# Single reconstruction and raw data

In [ ]:
print(d.shape, x_est.shape)
if SAVE_FIGS:
    y_path = f"{fig_path}/cdi_object_y_{pixel_intensity:.3g}.png"
    d_path = f"{fig_path}/cdi_sample_d_{pixel_intensity:.3g}.png"
else:
    y_path, d_path = None, None
plot_intensity_phase(
    fft2_shifted(x_true),
    use_log=True,
    label="y",
    save_path=y_path,
    intensity_unit="expected counts",
    x_label=FREQ_X_LABEL,
    y_label=FREQ_Y_LABEL,
    extent=k_extent,
    use_rad_extent=True,
)
plot(
    d[0, :],
    use_log=True,
    label="d (counts)",
    save_path=d_path,
    x_label=FREQ_X_LABEL,
    y_label=FREQ_Y_LABEL,
    extent=k_extent,
    use_rad_extent=True,
)

# $\ell$ plots

In [ ]:
from master.mle import log_likelihood, log_likelihood_x
from master.stats import mean_with_standard_error

USE_GMEAN = False


def plot_ll_convergence_fo(
    pixel_intensity=1e3,
    init="ORACLE",
    y_c_bottom=0.75,
    y_c_top=1.25,
):
    meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
    x_true, d, measurement_params, measurement_hash_meas = loader.get_measurement(
        meas_id
    )
    ell_d = np.asarray(log_likelihood(d, np.sqrt(d)))

    def norm_ll(lls):
        return (ell_d - lls) / ell_d

    N_alphas = 16
    test = False
    if test:
        alphas = [0.1]
    else:
        alphas = np.logspace(-4, 1, N_alphas)
    init_est_id = f"{init}-averaged"
    if init == "ORACLE":
        init_est = x_true
    else:
        init_est, _, _, _, _ = loader.get_estimate(meas_id, init_est_id)
    ell_init = np.asarray(log_likelihood_x(d, init_est))
    norm_ell_init = np.asarray(norm_ll(log_likelihood_x(d, init_est)))

    ll_upper_bound, _ = mean_with_standard_error(norm_ell_init, USE_GMEAN, -1)
    print(ell_init.shape)
    print(ll_upper_bound)
    min_ll = ll_upper_bound
    fig = plt.figure(figsize=(FIG_W, FIG_H), layout="compressed")
    for alpha in alphas:
        est_id = f"{init}-averaged-first-order-ml-{alpha:.3g}"
        # est_label = "ORACLE+FS"
        x_est, simulation_params, lls, config_hash, measurement_hash_est = (
            loader.get_estimate(meas_id, est_id)
        )
        # Idea: trivial maximum of dataset is y = sqrt(d).
        # Normalize using this to get somewhat invariant measure of convergence
        normed_ll = norm_ll(lls)
        mean_norm_ll, se_norm_ll = mean_with_standard_error(normed_ll, USE_GMEAN, -1)
        min_ll = np.minimum(np.nanmin(mean_norm_ll), min_ll)
        plt.errorbar(
            range(1, lls.shape[0] + 1),
            mean_norm_ll,
            yerr=se_norm_ll,
            label=rf"$\alpha = {alpha:.3g}$",
        )

    plt.xlabel(r"$i$")
    plt.ylabel(r"$\tilde{\ell}(\hat{x}^{(i)})$")
    plt.xscale("log")
    plt.yscale("log")
    plt.ylim(min_ll * y_c_bottom, ll_upper_bound * y_c_top)
    plt.xlim(left=1)
    plt.legend(loc="lower right")
    if SAVE_FIGS:
        suffix = "_gmean" if USE_GMEAN else ""
        save_path = f"{fig_path}/cdi-ell/ell_{init}{suffix}_{pixel_intensity:.3g}.png"
    else:
        save_path = None

    if save_path is not None:
        fig.savefig(save_path, dpi=PLOT_DPI)
    plt.show()

In [ ]:
plot_ll_convergence_fo(1e3, "HIO")

In [ ]:
plot_ll_convergence_fo(1e10, "HIO")

In [ ]:
plot_ll_convergence_fo(1e3, "ORACLE", 0.9, 1.1)
plot_ll_convergence_fo(1e10, "ORACLE", 0.9, 1.1)

# Estimator evaluation

In [ ]:
est_base_path = f"{fig_path}/estimator"


def evaluate_estimator_and_save_plots(
    x_model_ml, x_obj, obj_support, label="HIO+ML", extent=None
):
    make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
    n_hat = np.abs(x_model_ml) ** 2 - jnp.var(
        x_model_ml, axis=0, ddof=1, keepdims=True
    )  # / N_samples
    n = jnp.abs(x_obj) ** 2
    n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
    print("n_l")
    plot(
        n_hat_mu,
        label=r"$\operatorname{Mean}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_mean"),
    )
    # plot(n, label=r"$n_l$", extent=extent)
    plot(
        n_hat_bias,
        label=r"$\operatorname{Bias}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_bias"),
    )
    plot(
        n_hat_std,
        label=r"$\operatorname{Std}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_std"),
    )
    plot(
        n_hat_MSE,
        label=r"$\operatorname{MSE}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_MSE"),
    )

    print("phi_l")

    phi_hat = np.angle(x_model_ml) * obj_support
    phi = jnp.angle(x_obj) * obj_support

    phase_about_mean = True

    # Circular mean of the phase estimates
    phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))
    wrap = lambda a: np.angle(np.exp(1j * a))

    if phase_about_mean:
        # Deviations about the circular mean; bias = circular mean vs truth
        phi_hat_dev = wrap(phi_hat - phi_hat_mu)
        phi_hat_bias = wrap(phi_hat_mu - phi)
        phi_hat_std = np.std(phi_hat_dev, axis=0, ddof=1)
        phi_hat_MSE = phi_hat_bias**2 + phi_hat_std**2
    else:
        # Errors wrapped about the true phase
        phi_hat_error = wrap(phi_hat - phi)
        phi_hat_bias = np.mean(phi_hat_error, axis=0)
        phi_hat_std = np.std(phi_hat_error, axis=0, ddof=1)
        phi_hat_MSE = np.mean(phi_hat_error**2, axis=0)

    # # Circular mean of the phase estimates
    # phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))

    # # Wrapped phase errors in (-pi, pi]
    # phi_hat_error = np.angle(np.exp(1j * (phi_hat - phi)))

    # # Statistics in the local phase coordinate
    # phi_hat_bias = np.mean(phi_hat_error, axis=0)
    # phi_hat_std = np.std(phi_hat_error, axis=0, ddof=1)
    # phi_hat_MSE = np.mean(phi_hat_error**2, axis=0)

    plot(
        phi_hat_mu,
        label=r"$\operatorname{Mean}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_mean"),
    )
    plot(
        phi_hat_bias,
        label=r"$\operatorname{Bias}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_bias"),
    )
    plot(
        phi_hat_std,
        label=r"$\operatorname{Std}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_std"),
    )
    plot(
        phi_hat_MSE,
        label=r"$\operatorname{MSE}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_MSE"),
    )
    return n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias


n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias = (
    evaluate_estimator_and_save_plots(
        center_crop(x_est * T_factor, N_crop),
        center_crop(x_true * T_factor, N_crop),
        center_crop(obj_support, N_crop),
        label=est_label,
        extent=r_extent_cropped,
    )
)

In [ ]:
def evaluate_estimator_and_save_plots(
    x_model_ml, x_obj, obj_support, label="HIO+ML", extent=None
):
    make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
    M = x_model_ml.shape[0]

    n_hat = np.abs(x_model_ml) ** 2 - jnp.var(x_model_ml, axis=0, ddof=1, keepdims=True)
    n = jnp.abs(x_obj) ** 2
    n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
    n_hat_z = n_hat_bias / (n_hat_std / jnp.sqrt(M))

    print("n_l")
    plot(
        n_hat_mu,
        label=r"$\operatorname{Mean}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_mean"),
    )
    plot(
        n_hat_z,
        label=r"$\operatorname{Bias}(\hat{n}) / \operatorname{SE}$",
        extent=extent,
        save_path=make_path("nhat_bias_z"),
    )
    plot(
        n_hat_std,
        label=r"$\operatorname{Std}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_std"),
    )
    plot(
        n_hat_MSE,
        label=r"$\operatorname{MSE}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_MSE"),
    )

    print("phi_l")
    phi_hat = np.angle(x_model_ml) * obj_support
    phi = jnp.angle(x_obj) * obj_support
    wrap = lambda a: np.angle(np.exp(1j * a))

    phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))
    phi_hat_bias = wrap(phi_hat_mu - phi)
    phi_hat_std = np.std(wrap(phi_hat - phi_hat_mu), axis=0, ddof=1)
    phi_hat_MSE = phi_hat_bias**2 + phi_hat_std**2
    phi_hat_z = phi_hat_bias / (phi_hat_std / np.sqrt(M))

    plot(
        phi_hat_mu,
        label=r"$\operatorname{Mean}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_mean"),
    )
    plot(
        phi_hat_z,
        label=r"$\operatorname{Bias}(\hat{\phi}) / \operatorname{SE}$",
        extent=extent,
        save_path=make_path("phihat_bias_z"),
    )
    plot(
        phi_hat_std,
        label=r"$\operatorname{Std}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_std"),
    )
    plot(
        phi_hat_MSE,
        label=r"$\operatorname{MSE}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_MSE"),
    )

    return n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias


n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias = (
    evaluate_estimator_and_save_plots(
        center_crop(x_est * T_factor, N_crop),
        center_crop(x_true * T_factor, N_crop),
        center_crop(obj_support, N_crop),
        label=est_label,
        extent=r_extent_cropped,
    )
)

In [ ]:
# est_base_path = f"{fig_path}/estimator"


# def evaluate_estimator_and_save_plots(
#     x_model_ml, x_obj, obj_support, label="HIO+ML", extent=None
# ):
#     make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
#     n_hat = np.abs(x_model_ml) ** 2
#     n = jnp.abs(x_obj) ** 2
#     n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
#     print("n_l")
#     plot(
#         n_hat_mu,
#         label=r"$\operatorname{Mean}(\hat{n}_l)$",
#         extent=extent,
#         save_path=make_path("nhat_mean"),
#     )
#     # plot(n, label=r"$n_l$", extent=extent)
#     plot(
#         n_hat_bias,
#         label=r"$\operatorname{Bias}(\hat{n}_l)$",
#         extent=extent,
#         save_path=make_path("nhat_bias"),
#     )
#     plot(
#         n_hat_std,
#         label=r"$\operatorname{Std}(\hat{n}_l)$",
#         extent=extent,
#         save_path=make_path("nhat_std"),
#     )
#     plot(
#         n_hat_MSE,
#         label=r"$\operatorname{MSE}(\hat{n}_l)$",
#         extent=extent,
#         save_path=make_path("nhat_MSE"),
#     )

#     print("phi_l")

#     phi_hat = np.angle(x_model_ml) * obj_support
#     phi = jnp.angle(x_obj) * obj_support

#     # Circular mean of the phase estimates
#     phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))

#     # Wrapped phase errors in (-pi, pi]
#     phi_hat_error = np.angle(np.exp(1j * (phi_hat - phi)))

#     # Statistics in the local phase coordinate
#     phi_hat_bias = np.mean(phi_hat_error, axis=0)
#     phi_hat_std = np.std(phi_hat_error, axis=0, ddof=1)
#     phi_hat_MSE = np.mean(phi_hat_error**2, axis=0)

#     plot(
#         phi_hat_mu,
#         label=r"$\operatorname{Mean}(\hat{\phi}_l)$",
#         extent=extent,
#         save_path=make_path("phihat_mean"),
#     )
#     plot(
#         phi_hat_bias,
#         label=r"$\operatorname{Bias}(\hat{\phi}_l)$",
#         extent=extent,
#         save_path=make_path("phihat_bias"),
#     )
#     plot(
#         phi_hat_std,
#         label=r"$\operatorname{Std}(\hat{\phi}_l)$",
#         extent=extent,
#         save_path=make_path("phihat_std"),
#     )
#     plot(
#         phi_hat_MSE,
#         label=r"$\operatorname{MSE}(\hat{\phi}_l)$",
#         extent=extent,
#         save_path=make_path("phihat_MSE"),
#     )
#     return n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias


# n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias = (
#     evaluate_estimator_and_save_plots(
#         center_crop(x_est * T_factor, N_crop),
#         center_crop(x_true * T_factor, N_crop),
#         center_crop(obj_support, N_crop),
#         label=est_label,
#         extent=r_extent_cropped,
#     )
# )

In [ ]:
# Verify that all plots are the same resolution
d = r"~/master/docs/thesis/figures/estimator/ORACLE+FS"
!file {d}/*.png

In [ ]:
y_ph = fft2_shifted(x_true)
N_y = y_ph.size
n_l_std_bouchet, phi_l_std_bouchet = bouchet_bounds(x_true)
plot(
    center_crop(n_l_std_bouchet * (T_factor) ** 2, N_crop),
    label=r"$[\sigma_{\hat{n}_l}]^{\text{Bouchet}}$",
    extent=r_extent_cropped,
)
plot(
    center_crop(phi_l_std_bouchet, N_crop),
    label=r"$[\sigma_{\hat{\phi}_l}]^{Bouchet}$",
    extent=r_extent_cropped,
)

In [ ]:
Jn_diag, Jphi_diag, Jnphi_off, n_var_diag, phi_var_diag = fisher_2x2_block_fast(
    x_true, y_ph, N_y
)


def diag_of_2x2_inverse(a, b, c, d):
    denom = a * d - b * c
    return d / (denom), a / denom


Jn_inv_2x2, Jphi_2x2 = diag_of_2x2_inverse(Jn_diag, Jnphi_off, Jnphi_off, Jphi_diag)
n_l_std_2x2_crb = jnp.sqrt(Jn_inv_2x2) * obj_support * T_factor**2
phi_l_std_2x2_crb = jnp.sqrt(Jphi_2x2) * obj_support

# In practice, nphi coupling is zero! Relevant property
print(np.mean(Jn_diag * Jphi_diag), np.mean(Jnphi_off**2))

label_submatrix = r"\text{pixel}"
label_pinv = r"\text{EWCRB}"
label_RPA = r"\text{RPA}"
print(pixel_intensity)
plot(
    center_crop(n_l_std_2x2_crb, N_crop),
    label=r"$\sigma^{" + label_submatrix + r"}_{\hat{n}}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_n_2x2_inv_{pixel_intensity:.3g}.png",
)
plot(
    center_crop(phi_l_std_2x2_crb, N_crop),
    label=r"$\sigma^{" + label_submatrix + r"}_{\hat{\phi}}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_phi_2x2_inv_{pixel_intensity:.3g}.png",
)
# Visualize to show that per-pixel information is perfectly uniform in this parametrization
# plot(
#     det_j * obj_support,
#     label=r"$[det(J_{jj})]$",
# )

In [ ]:
Jpinv_nn, Jpinv_phiphi, Jpinv_nphi = fisher_pinv_2x2_block(x_true, y_ph, N_y)

n_std_crb = jnp.sqrt(Jpinv_nn) * obj_support * T_factor**2
phi_std_crb = jnp.sqrt(Jpinv_phiphi) * obj_support

plot(
    center_crop(n_std_crb, N_crop),
    label=r"$\sigma^{" + label_pinv + r"}_{\hat{n}}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_n_pinv_CRB_{pixel_intensity:.3g}.png",
)
plot(
    center_crop(phi_std_crb, N_crop),
    label=r"$\sigma^{" + label_pinv + r"}_{\hat{\phi}}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_phi_pinv_CRB_{pixel_intensity:.3g}.png",
)

In [ ]:
Jrpa_nn, Jrpa_phiphi = fisher_random_phase(x_true)[-2:]

n_std_crb = jnp.sqrt(Jrpa_nn) * obj_support * T_factor**2
phi_std_crb = jnp.sqrt(Jrpa_phiphi) * obj_support

plot(
    center_crop(n_std_crb, N_crop),
    label=r"$\sigma^{" + label_RPA + r"}_{\hat{n}}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_n_random_phase_{pixel_intensity:.3g}.png",
)
plot(
    center_crop(phi_std_crb, N_crop),
    label=r"$\sigma^{" + label_RPA + r"}_{\hat{\phi}}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_phi_random_phase_{pixel_intensity:.3g}.png",
)

In [ ]:
def ratio_plot(
    theta_hat_std, theta_std_bound, theta_label=r"n", bound_label=label_submatrix
):
    plot(
        theta_hat_std / center_crop(theta_std_bound, N_crop),
        label=r"$\operatorname{Std}[\hat{"
        + theta_label
        + r"}_l] / \sigma_{\hat{"
        + theta_label
        + r"}_l}^{"
        + bound_label
        + r"}$",
        extent=r_extent_cropped,
    )


ratio_plot(n_hat_std, n_l_std_2x2_crb, r"n", label_submatrix)
ratio_plot(phi_hat_std, phi_l_std_2x2_crb, r"\phi", label_submatrix)

In [ ]:
ratio_plot(n_hat_std, n_std_crb, r"n", label_pinv)
ratio_plot(phi_hat_std, phi_std_crb, r"\phi", label_pinv)

# Repeat in $u = \Re{x}, v = \Im{x}$ parametrization

In [ ]:
est_base_path = f"{fig_path}/estimator"


def evaluate_estimator_Re_Im_and_save_plots(
    x_model_ml, x_obj, obj_support, factor, label="HIO+ML", extent=None
):
    make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
    u_hat = np.real(x_model_ml)
    v_hat = np.imag(x_model_ml)
    u = np.real(x_obj)
    v = np.imag(x_obj)
    # Re x / u block.

    u_hat_mu, u_hat_std, u_hat_bias, u_hat_MSE = estimator_stats(u_hat, u)
    print("u_l")
    plot(
        u_hat_mu * factor,
        label=r"$\operatorname{Mean}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_mean"),
    )
    plot(
        u_hat_bias * factor,
        label=r"$\operatorname{Bias}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_bias"),
    )
    plot(
        u_hat_std * factor,
        label=r"$\operatorname{Std}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_std"),
    )
    plot(
        u_hat_MSE * factor**2,
        label=r"$\operatorname{MSE}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_MSE"),
    )

    # Im x / v block.
    v_hat_mu, v_hat_std, v_hat_bias, v_hat_MSE = estimator_stats(v_hat, v)
    print("v_l")
    plot(
        v_hat_mu * factor,
        label=r"$\operatorname{Mean}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_mean"),
    )
    plot(
        v_hat_bias * factor,
        label=r"$\operatorname{Bias}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_bias"),
    )
    plot(
        v_hat_std * factor,
        label=r"$\operatorname{Std}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_std"),
    )
    plot(
        v_hat_MSE * factor**2,
        label=r"$\operatorname{MSE}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_MSE"),
    )

    return (
        u_hat_mu,
        u_hat_std,
        u_hat_bias,
        u_hat_MSE,
        v_hat_mu,
        v_hat_std,
        v_hat_bias,
        v_hat_MSE,
    )


(
    u_hat_mu,
    u_hat_std,
    u_hat_bias,
    u_hat_MSE,
    v_hat_mu,
    v_hat_std,
    v_hat_bias,
    v_hat_MSE,
) = evaluate_estimator_Re_Im_and_save_plots(
    center_crop(x_est, N_crop),
    center_crop(x_true, N_crop),
    center_crop(obj_support, N_crop),
    T_factor,
    label=est_label,
    extent=r_extent_cropped,
)

# Compare CRB from $J_x^+$ with lower bound from $1/\operatorname{diag}(J_x)$

In [ ]:
Jpinv_uu, Jpinv_vv, Jpinv_uv = fisher_pinv_2x2_block_cart(x_true, y_ph, N_y)

x_u_std_crb = jnp.sqrt(Jpinv_uu) * obj_support
x_v_std_crb = jnp.sqrt(Jpinv_vv) * obj_support

plot(
    center_crop(x_u_std_crb, N_crop) * T_factor,
    label=r"$\sigma_{\hat{u}}^{" + label_pinv + r"}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_u_pinv_CRB_{pixel_intensity:.3g}.png",
)
plot(
    center_crop(x_v_std_crb, N_crop) * T_factor,
    label=r"$\sigma_{\hat{v}}^{" + label_pinv + r"}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_v_pinv_CRB_{pixel_intensity:.3g}.png",
)
# Visualize to show that per-pixel information is perfectly uniform in this parametrization
# plot(
#     det_j * obj_support,
#     label=r"$[det(J_{jj})]$",
# )

In [ ]:
Juu, Jvv, Juv, _, _ = fisher_2x2_block_fast_cart(x_true, y_ph, N_y)
Ju_inv_2x2, Jv_inv_2x2 = diag_of_2x2_inverse(Juu, Juv, Juv, Jvv)

# In practice, nphi coupling is zero! Relevant property
print(np.mean(Juu * Jvv), np.mean(Juv**2))

x_u_std_diag = jnp.sqrt(Ju_inv_2x2) * obj_support
x_v_std_diag = jnp.sqrt(Jv_inv_2x2) * obj_support
plot(
    center_crop(x_u_std_diag, N_crop) * T_factor,
    label=r"$\sigma_{\hat{u}}^{" + label_submatrix + r"}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_u_2x2_inv_{pixel_intensity:.3g}.png",
)
plot(
    center_crop(x_v_std_diag, N_crop) * T_factor,
    label=r"$\sigma_{\hat{v}}^{" + label_submatrix + r"}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_v_2x2_inv_{pixel_intensity:.3g}.png",
)

In [ ]:
Jrpa_uu, Jrpa_vv = fisher_random_phase_cart(x_true)[-2:]
x_u_std_diag = jnp.sqrt(Jrpa_uu) * obj_support
x_v_std_diag = jnp.sqrt(Jrpa_vv) * obj_support
plot(
    center_crop(x_u_std_diag, N_crop) * T_factor,
    label=r"$\sigma_{\hat{u}}^{" + label_RPA + r"}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_u_random_phase_{pixel_intensity:.3g}.png",
)
plot(
    center_crop(x_v_std_diag, N_crop) * T_factor,
    label=r"$\sigma_{\hat{v}}^{" + label_RPA + r"}$",
    extent=r_extent_cropped,
    save_path=f"{fig_path}/bounds/cdi_v_random_phase_{pixel_intensity:.3g}.png",
)

In [ ]:
ratio_plot(u_hat_std, x_u_std_diag, r"u", label_submatrix)
ratio_plot(v_hat_std, x_v_std_diag, r"v", label_submatrix)

In [ ]:
ratio_plot(u_hat_std, x_u_std_crb, r"u", label_pinv)
ratio_plot(v_hat_std, x_v_std_crb, r"v", label_pinv)

In [ ]:
def bound_ratio_plot(
    theta_std_bound_1,
    theta_std_bound_2,
    theta_label="n",
    bound_label_1=label_pinv,
    bound_label_2=label_submatrix,
):
    plot(
        center_crop(theta_std_bound_1, N_crop) / center_crop(theta_std_bound_2, N_crop),
        label=r"$\sigma_{\hat{"
        + theta_label
        + r"}}^{"
        + bound_label_1
        + r"} / \sigma_{\hat{"
        + theta_label
        + r"}}^{"
        + bound_label_2
        + r"}$",
        extent=r_extent_cropped,
    )


bound_ratio_plot(x_u_std_diag, x_u_std_crb, r"u")
bound_ratio_plot(x_v_std_diag, x_v_std_crb, r"v")